In [ ]:
!pip install langchain langchain-community langchain-core langchain-text-splitters chromadb pypdf sentence-transformers

In [ ]:
!pip install -U langchain-text-splitters

In [ ]:
!pip install -U langchain-community langchain-google-genai langchain chromadb pypdf langgraph

In [ ]:
!pip install --upgrade -q langchain-google-genai langchain-community chromadb

In [ ]:
!pip install --upgrade -q langchain-google-genai langchain-community chromadb

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# PDF load + split

In [ ]:
import os
from typing import TypedDict, List
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# २. State
class AgentState(TypedDict):
    question: str
    answer: str
    context: List[str]
    needs_human: bool

# ३. Retrieval
def retrieve_docs(state: AgentState):
    print("--- LOG: STARTING RETRIEVAL ---")
    pdf_path = "RAG_Project_Documentation.pdf"
    if not os.path.exists(pdf_path):
        pdf_path = "RAG_Project_Documentation.pdf.pdf"

    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = text_splitter.split_documents(documents)

    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vectorstore = Chroma.from_documents(chunks, embeddings)
    return {"context": vectorstore.as_retriever(search_kwargs={"k": 2}).invoke(state["question"])}

# ४. Generation
def generate_answer(state: AgentState):
    print("--- LOG: GENERATING RESPONSE ---")
    if not state["context"]:
        return {"answer": "No info found.", "needs_human": True}


    context_data = state["context"][0].page_content[:500]
    final_ans = f"Based on the document: {context_data}"

    return {"answer": final_ans, "needs_human": False}

# ५. Graph Setup
from langgraph.graph import StateGraph, END
workflow = StateGraph(AgentState)
workflow.add_node("retriever", retrieve_docs)
workflow.add_node("generator", generate_answer)
workflow.set_entry_point("retriever")
workflow.add_edge("retriever", "generator")
workflow.add_edge("generator", END)
app = workflow.compile()

# ६. Run
result = app.invoke({"question": "What are the features?"})
print("\n" + "="*50)
print(f"ASSISTANT: {result['answer']}")
print(f"HUMAN ESCALATION: {result['needs_human']}")
print("="*50)

In [ ]:
# Test

In [ ]:
def ask(user_question: str):
    # This block automatically checks which variable name you used for the graph
    try:
        # Try using 'customer_support_app'
        response = customer_support_app.invoke({"question": user_question})
    except NameError:
        # If that fails, try using 'app'
        response = app.invoke({"question": user_question})

    return response['answer']

# --- RUN THE TEST AGAIN ---
if __name__ == "__main__":
    print("\n" + "="*60)
    print("       CUSTOMER SUPPORT AI AGENT - PRODUCTION TEST")
    print("="*60)

    # Try asking about the architecture instead of refund
query_final = "What are the architecture components?"
print(f"\n[USER]: {query_final}")
print(f"[AI]:   {ask(query_final)}")